# F3-10k: Qwen2.5-VL-3B + QLoRA rank 16

F2 оказалась лучшим из обученных адаптеров: она сохранила строгие метрики baseline и получила BERTScore F1 85,11%, немного опередив F1-fast. В F3 сохраняется архитектура Qwen2.5-VL-3B и LoRA rank 16, а объём обучения увеличивается с 2 000 до 10 000 примеров VK.

Адаптер обучается заново от базовой Qwen, чтобы результат не зависел от аварийного восстановления F2. Learning rate снижен с `2e-4` до `1e-4`. На RTX 4060 ожидается 1 250 шагов и около 8–10 часов обучения. Чекпойнты сохраняются каждые 100 шагов, поэтому запуск можно продолжить после перезагрузки.

In [ ]:
import json
import math
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
import requests
import torch
import urllib3
from PIL import Image
from torch.utils.data import Dataset
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

os.environ['PYTHONIOENCODING'] = 'utf-8'
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DATA = ROOT / 'data' / 'raw' / 'llava_instruct_ru_train.json'
SUBSET_PATH = ROOT / 'data' / 'processed' / 'llava_instruct_ru_first_iteration_seed42.csv'
IMAGE_CACHE = ROOT / 'data' / 'coco_cache' / 'train2017'
MODEL_DIR = ROOT / 'models' / 'f3_10k_qwen_r16_adapter'
CHECKPOINT_DIR = ROOT / 'checkpoints' / 'f3_10k_qwen_r16'
RESULTS_DIR = ROOT / 'results'
for directory in (IMAGE_CACHE, MODEL_DIR, CHECKPOINT_DIR, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
SEED = 42
TRAINING_EXAMPLES = 10_000
MAX_LENGTH = 1024
GRADIENT_ACCUMULATION_STEPS = 8
assert RAW_DATA.exists() and SUBSET_PATH.exists(), 'Не найдены подготовленные данные из ноутбука 01.'
assert torch.cuda.is_available(), 'Для обучения нужна CUDA.'
print('GPU:', torch.cuda.get_device_name(0))

## 1. Расширенная обучающая выборка VK

Используются 10 000 примеров из зафиксированной стратифицированной выборки, подготовленной в ноутбуке 01. Соотношение типов инструкций сохраняется, а `seed=42` обеспечивает воспроизводимость.

In [ ]:
raw_records = json.loads(RAW_DATA.read_text(encoding='utf-8'))
subset = pd.read_csv(SUBSET_PATH)

def parse_record(record):
    messages = record['conversations']
    question = next(item['value'] for item in messages if item['from'] == 'human')
    answer = next(item['value'] for item in messages if item['from'] == 'gpt')
    return {
        'type': record.get('type', 'unknown'),
        'question': question.replace('<image>\n', '').strip(),
        'answer': answer.strip(),
        'image_path': record['image'],
    }

candidate_rows = []
for row in subset[['row_number', 'type']].itertuples(index=False):
    parsed = parse_record(raw_records[int(row.row_number)])
    parsed['type'] = row.type
    candidate_rows.append(parsed)
candidate_df = pd.DataFrame(candidate_rows)

type_counts = candidate_df['type'].value_counts()
allocation = (type_counts / len(candidate_df) * TRAINING_EXAMPLES).round().astype(int)
allocation.iloc[-1] += TRAINING_EXAMPLES - allocation.sum()
train_df = pd.concat([
    group.sample(n=allocation[type_name], random_state=SEED)
    for type_name, group in candidate_df.groupby('type')
]).sample(frac=1, random_state=SEED).reset_index(drop=True)
assert len(train_df) == TRAINING_EXAMPLES
TOTAL_STEPS = math.ceil(len(train_df) / GRADIENT_ACCUMULATION_STEPS)
WARMUP_STEPS = round(TOTAL_STEPS * 0.03)
print(f'Обучающих примеров: {len(train_df):,}'.replace(',', ' '))
print(f'Уникальных изображений: {train_df.image_path.nunique():,}'.replace(',', ' '))
print(f'Оптимизационных шагов: {TOTAL_STEPS:,}; warmup: {WARMUP_STEPS}'.replace(',', ' '))
display(train_df['type'].value_counts().rename_axis('type').reset_index(name='examples'))

## 2. Проверка изображений COCO

Изображения для исходной 12-тысячной выборки уже загружались ранее, поэтому обычно скачивание не потребуется. Если отдельные файлы отсутствуют, ячейка догрузит только их и выведет прогресс.

In [ ]:
def coco_url(image_path):
    parts = Path(image_path).parts
    split, filename = parts[-2], parts[-1]
    return f'https://images.cocodataset.org/{split}/{filename}'

required_images = sorted(train_df.image_path.unique())
missing_images = [path for path in required_images if not (IMAGE_CACHE / Path(path).name).exists()]
print(f'Всего уникальных изображений: {len(required_images)}')
print(f'Уже в кэше: {len(required_images) - len(missing_images)}; нужно скачать: {len(missing_images)}')

def download_image(image_path):
    target = IMAGE_CACHE / Path(image_path).name
    if target.exists():
        return True, image_path, 'cached'
    last_error = None
    for attempt in range(3):
        try:
            response = requests.get(coco_url(image_path), timeout=(15, 60), verify=False)
            response.raise_for_status()
            target.write_bytes(response.content)
            return True, image_path, 'downloaded'
        except Exception as error:
            last_error = error
            time.sleep(1 + attempt)
    return False, image_path, str(last_error)

failed = []
if missing_images:
    started = time.perf_counter()
    with ThreadPoolExecutor(max_workers=8) as executor:
        futures = [executor.submit(download_image, path) for path in missing_images]
        for completed, future in enumerate(as_completed(futures), start=1):
            ok, image_path, status = future.result()
            if not ok:
                failed.append((image_path, status))
            if completed % 100 == 0 or completed == len(futures):
                elapsed = (time.perf_counter() - started) / 60
                print(f'Готово {completed}/{len(futures)} | ошибок: {len(failed)} | {elapsed:.1f} мин.')

assert not failed, f'Не удалось скачать {len(failed)} изображений. Включите VPN и повторите ячейку.'
print('Все изображения выборки F3-10k доступны локально.')

## 3. Датасет и QLoRA rank 16

In [ ]:
class VkFullDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        return self.frame.iloc[index].to_dict()

def load_local_image(image_path):
    return Image.open(IMAGE_CACHE / Path(image_path).name).convert('RGB')

dataset = VkFullDataset(train_df)
quantization = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
)
processor = AutoProcessor.from_pretrained(
    MODEL_ID, min_pixels=256 * 28 * 28, max_pixels=512 * 28 * 28,
)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=quantization, device_map='auto', dtype=torch.float16,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
))
model.print_trainable_parameters()

In [ ]:
class VlmDataCollator:
    def __call__(self, features):
        images = [load_local_image(item['image_path']) for item in features]
        full_texts, prompt_texts = [], []
        for item, image in zip(features, images):
            user = {'role': 'user', 'content': [
                {'type': 'image', 'image': image},
                {'type': 'text', 'text': item['question']},
            ]}
            full = [user, {'role': 'assistant', 'content': [{'type': 'text', 'text': item['answer']}]}]
            full_texts.append(processor.apply_chat_template(full, tokenize=False))
            prompt_texts.append(processor.apply_chat_template(
                [user], tokenize=False, add_generation_prompt=True))
        batch = processor(
            text=full_texts, images=images, padding=True, truncation=True,
            max_length=MAX_LENGTH, return_tensors='pt',
        )
        prompt_batch = processor(
            text=prompt_texts, images=images, padding=True, truncation=True,
            max_length=MAX_LENGTH, return_tensors='pt',
        )
        labels = batch['input_ids'].clone()
        labels[batch['attention_mask'] == 0] = -100
        for i, prompt_length in enumerate(prompt_batch['attention_mask'].sum(dim=1).tolist()):
            labels[i, :prompt_length] = -100
        batch['labels'] = labels
        return batch

collator = VlmDataCollator()

## 4. Обучение F3-10k с автоматическим продолжением

При повторном запуске выбирается последний полноценный чекпойнт, содержащий веса, optimizer, scheduler и состояние Trainer. Повреждённые незавершённые каталоги автоматически игнорируются.

In [ ]:
def latest_complete_checkpoint(directory):
    candidates = []
    for path in directory.glob('checkpoint-*'):
        required = ['adapter_model.safetensors', 'optimizer.pt', 'scheduler.pt', 'trainer_state.json']
        if all((path / name).exists() for name in required):
            candidates.append(path)
    return max(candidates, key=lambda path: int(path.name.split('-')[-1]), default=None)

resume_checkpoint = latest_complete_checkpoint(CHECKPOINT_DIR)
training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR), num_train_epochs=1,
    per_device_train_batch_size=1, gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=1e-4, lr_scheduler_type='cosine', warmup_steps=WARMUP_STEPS,
    logging_steps=10, save_strategy='steps', save_steps=100, save_total_limit=3,
    fp16=True, gradient_checkpointing=True, optim='paged_adamw_8bit',
    report_to='none', remove_unused_columns=False, seed=SEED,
)
trainer = Trainer(model=model, args=training_args, train_dataset=dataset, data_collator=collator)
print(f'Всего шагов: {TOTAL_STEPS}; чекпойнт каждые 100 шагов.')
if resume_checkpoint:
    print(f'Продолжаю из {resume_checkpoint.name}.')
else:
    print('Начинаю F3-10k с базовой модели.')
trainer.train(resume_from_checkpoint=str(resume_checkpoint) if resume_checkpoint else None)

model.save_pretrained(MODEL_DIR)
processor.save_pretrained(MODEL_DIR)
(RESULTS_DIR / 'f3_10k_training_config.json').write_text(json.dumps({
    'base_model': MODEL_ID, 'examples': len(dataset), 'lora_rank': 16,
    'epochs': 1, 'learning_rate': 1e-4, 'seed': SEED,
    'total_steps': TOTAL_STEPS, 'warmup_steps': WARMUP_STEPS,
}, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Адаптер F3-10k сохранён в: {MODEL_DIR}')